## **Diagnóstico de cancro da mama com CNNs e Active Learning**
---
Inteligência Artificial Centrada no Humano

Faculdade de Ciências e Tecnologia da Universidade de Coimbra | Mestrado em Engenharia e Ciência de Dados

Compara três estratégias de query do modAL (`uncertainty`, `entropy`, `margin`) com vários batch sizes. Correr primeiro `01_passive_learning.ipynb` para ter a baseline.

In [ ]:
# Configuração do ambiente: funciona no Google Colab e localmente
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/miacd-iach-breast-cancer-diagnosis")  # pasta do repositório no Drive
    %pip install -q modAL-python scikeras
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / "data/breast-ultrasound-images-DatasetNinja/ds/img"
RESULTS_PATH = ROOT / "results"
RESULTS_PATH.mkdir(exist_ok=True)

In [ ]:
# Parâmetros
IMG_SIZE = 224          # reduzir (ex.: 128) se faltar memória
MAX_PER_CLASS = None    # limitar o nº de imagens originais por classe (None = todas)
STRATEGIES = ["uncertainty", "entropy", "margin"]
BATCH_SIZES = [16, 32, 64, 128]
N_INITIAL = 1000        # nº de imagens anotadas no início
N_QUERIES = 10          # nº de iterações de Active Learning (imagens pedidas ao humano)
EPOCHS = 10
HUMAN = False           # True: pede a label a uma pessoa; False: simula o humano com a label do dataset

### Dados

In [ ]:
# Carregar, dividir (treino/teste) e balancear o conjunto de treino
import numpy as np
from src.data import CLASSES, load_dataset

x_train, x_test, y_train, y_test = load_dataset(DATA_PATH, size=IMG_SIZE, max_per_class=MAX_PER_CLASS)

print(f"Conjunto de treino: {x_train.shape}, conjunto de teste: {x_test.shape}")
print(f"Memória das imagens: {(x_train.nbytes + x_test.nbytes) / 1e6:.0f} MB")
for split, y in [("Treino", y_train), ("Teste", y_test)]:
    print(split, {name: int(np.sum(y == i)) for i, name in enumerate(CLASSES)})

### Experiências

In [ ]:
from src.active_learning import run_active_learning
from src.evaluation import compute_metrics, load_json, plot_history, save_json

baseline_file = RESULTS_PATH / "passive_learning.json"
baseline = load_json(baseline_file) if baseline_file.exists() else None

results = {}   # results[estratégia][batch_size] = {"history", "y_pred", "metrics"}
for strategy in STRATEGIES:
    results[strategy] = {}
    for batch_size in BATCH_SIZES:
        print(f"{strategy} - batch {batch_size}")
        history, y_pred = run_active_learning(strategy, x_train, y_train, x_test, y_test,
                                              batch_size=batch_size, epochs=EPOCHS, n_initial=N_INITIAL,
                                              n_queries=N_QUERIES, human=HUMAN)
        results[strategy][batch_size] = {"history": history, "y_pred": y_pred.tolist(),
                                         "metrics": compute_metrics(y_test, y_pred)}

save_json(results, RESULTS_PATH / "active_learning.json")

### Resultados

In [ ]:
for strategy in STRATEGIES:
    histories = {batch: r["history"] for batch, r in results[strategy].items()}
    plot_history(histories, f"Active Learning using {strategy}_sampling", baseline=baseline)

In [ ]:
# Métricas no conjunto de teste (FN = maligno classificado como benigno/normal)
print(f"{'estratégia':<12}{'batch':>6}{'accuracy':>10}{'recall':>8}{'F1':>8}{'recall mal.':>13}{'FN':>5}")
if baseline is not None:
    m = baseline
    print(f"{'passive':<12}{'-':>6}{m['accuracy']:>10.3f}{m['recall_macro']:>8.3f}{m['f1_macro']:>8.3f}{m['recall_malignant']:>13.3f}{m['false_negatives']:>5}")
for strategy in STRATEGIES:
    for batch_size, r in results[strategy].items():
        m = r["metrics"]
        print(f"{strategy:<12}{batch_size:>6}{m['accuracy']:>10.3f}{m['recall_macro']:>8.3f}{m['f1_macro']:>8.3f}{m['recall_malignant']:>13.3f}{m['false_negatives']:>5}")

In [ ]:
# Matrizes de confusão da estratégia com menos falsos negativos
from src.evaluation import plot_confusion_matrix

best = min(STRATEGIES, key=lambda s: min(r["metrics"]["false_negatives"] for r in results[s].values()))
for batch_size, r in results[best].items():
    plot_confusion_matrix(y_test, r["y_pred"], title=f"{best}_sampling - batch {batch_size}")